**Additional note**

In [3]:
'''Imports setup'''
#!/usr/bin/env python3
import sys
import logging
from src.data_loader import DataLoader
from sklearn.model_selection import train_test_split

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

# Automatically finds the project root 'sira' and adds it to Python's path
PROJECT_ROOT = r"c:\Users\M.faisal\Desktop\sira"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
loader = DataLoader()
df = loader.load_data()
df.head()

INFO: Loading dataset...
INFO: 
Data Loaded Succesfully...


,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
0,1,pressure leak detected on pipeline 7 during ro...,Platform B,Aisha,Instrumentation,High,Leak,2025-09-30,Day,Open
1,4,smoke observed from electrical control panel,Refinery,Joy,Production,High,Electrical,2026-01-22,Day,Open
2,6,worker slipped on wet surface near loading bay,Flow Station,Blessing,Pipeline Operations,Medium,Slip/Fall,2025-10-26,Night,Resolved
3,7,oil spill discovered near storage tank,Wellhead 12,Paul,Pipeline Operations,High,Oil Spill,2025-11-09,Night,Resolved
4,8,corrosion observed on external pipeline coating,Control Room,Chinedu,Electrical,Medium,Corrosion,2025-11-19,Day,Resolved


### 1. train/test split with `FeatureEngineer`

This is what we should use in SIRA.
Add this import at the top `from sklearn.model_selection import train_test_split`.

In [4]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["report_text"],
    df["incident_type"],
    test_size=0.2,
    random_state=42,
    stratify=df["incident_type"]
)

Now we should have:

```text
800 training reports
200 testing reports
```

---

In [6]:
print(X_train_text.shape)

(739,)


In [9]:
print(X_test_text.shape)

(185,)


---

### 2.1. fit TF-IDF only on training data

```python
feature_engineer = FeatureEngineer(
    method="tfidf"
)

X_train = feature_engineer.fit_transform(
    X_train_text
)
```

This learns the vocabulary from the training data. Let's implement it.

---

In [10]:
from src.feature_engineering import FeatureEngineer

In [11]:
feature_engineer = FeatureEngineer(
    method="tfidf"
)

X_train = feature_engineer.fit_transform(
    X_train_text
)

### 2.2. Transform test data

```python
X_test = feature_engineer.transform(
    X_test_text
)
```

Notice that:

```python
fit_transform()
```

was used for training. But:

```python
transform()
```

was used for testing. This is critical.

---

In [12]:
X_test = feature_engineer.transform(
    X_test_text
)

### 3. Check `dimensions`

```python
print("Training shape:", X_train.shape)

print("Testing shape:", X_test.shape)
```

For example:

```text
Training shape: (800, 75)
Testing shape: (200, 75)
```

Notice something important:

Both have:

```text
75 features
```

Why?

Because they use the **same vocabulary**.

---

In [13]:
print("Training shape:", X_train.shape)

print("Testing shape:", X_test.shape)

Training shape: (739, 75)
Testing shape: (185, 75)


### 4. What happens to an `unknown word?`

Suppose training data contains: ```text gas leak detected ``` but a new report says:

```text
compressor vibration detected
```

If:

```text
compressor
vibration
```

were not present in the training vocabulary, the vectorizer doesn't suddenly create new columns. It transforms the text using the vocabulary it already learned. This is exactly what we want in production.

---

### 5. Switching to `Bag of Words`

The `FeatureEngineer` class makes this simple.

Instead of:

```python
feature_engineer = FeatureEngineer(
    method="tfidf"
)
```

we use:

```python
feature_engineer = FeatureEngineer(
    method="bow"
)
```

Then:

```python
X_train = feature_engineer.fit_transform(
    X_train_text
)

X_test = feature_engineer.transform(
    X_test_text
)
```

Everything else stays the same. That's one of the advantages of abstraction.

---

In [14]:
feature_engineer = FeatureEngineer(
    method="bow"
)

In [15]:
X_train = feature_engineer.fit_transform(
    X_train_text
)

X_test = feature_engineer.transform(
    X_test_text
)

### 6. Comparing both

We can run:

```python
tfidf_engineer = FeatureEngineer(
    method="tfidf"
)

X_train_tfidf = tfidf_engineer.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf_engineer.transform(
    X_test_text
)
```

Then:

```python
bow_engineer = FeatureEngineer(
    method="bow"
)

X_train_bow = bow_engineer.fit_transform(
    X_train_text
)

X_test_bow = bow_engineer.transform(
    X_test_text
)
```

Now we have:

```text
TF-IDF representation
        vs
Bag-of-Words representation
```

We can train the same classifier on both and compare performance.

---


# 26. What About `ngram_range`?

This is another important feature of your class.

Currently:

```python
ngram_range=(1, 1)
```

means:

> Use individual words.

These are called **unigrams**.

For:

```text
gas leak detected
```

you get:

```text
gas
leak
detected
```

---

With:

```python
ngram_range=(1, 2)
```

you get:

### Unigrams

```text
gas
leak
detected
```

### Bigrams

```text
gas leak
leak detected
```

This can be very useful in incident classification because phrases can contain more meaning than individual words.

For example:

```text
pressure leak
gas detection
equipment failure
oil spill
```

are potentially more informative than isolated words.

So:

```python
feature_engineer = FeatureEngineer(
    method="tfidf",
    ngram_range=(1, 2)
)
```

is a good experiment for SIRA.

---

### 7. `max_features`

Another useful parameter:

```python
max_features=5000
```

means:

> Don't allow the vectorizer to create more than 5,000 features.

For example:

```python
feature_engineer = FeatureEngineer(
    method="tfidf",
    max_features=5000,
    ngram_range=(1, 2)
)
```

This can control memory usage and model complexity.

---

### 8. The complete `SIRA` flow

By now you should ultimately understand this entire chain:

```text
                 CSV
                  │
                  ▼
              DataLoader
                  │
                  ▼
             Raw DataFrame
                  │
                  ▼
           Preprocessor
                  │
                  ▼
          Clean DataFrame
                  │
                  ▼
          Train/Test Split
                  │
           ┌──────┴──────┐
           ▼             ▼
       Training         Test
           │             │
           ▼             │
    fit_transform()      │
           │             │
           ▼             │
       TF-IDF/BOW        │
           │             │
           └──────┬──────┘
                  │
                  ▼
          Machine Learning
              Model
                  │
                  ▼
             Prediction
```

---

### 9. The most important distinction

A. `fit_transform()`

Means:

> **Learn the representation and transform the data.**

Use primarily on:

```text Training data ```

B. `transform()`

Means:

> **Use the representation already learned to transform new data.**

Use on:

```text
- Validation data
- Test data
- Production data
- New incident reports
```

Therefore:

```python
X_train = vectorizer.fit_transform(X_train_text)

X_test = vectorizer.transform(X_test_text)
```

is correct. Whereas:

```python
X_train = vectorizer.fit_transform(X_train_text)

X_test = vectorizer.fit_transform(X_test_text)
```

is **not** the correct pattern for an ML pipeline.

---

### 10. Why this matters when SIRA goes to production?

Eventually your SIRA application will receive:

```text
"High pressure detected around Pipeline 12"
```

The application cannot create a brand-new TF-IDF vocabulary every time a user submits a report.

Instead:

```text
TRAINING
   │
   ▼
TF-IDF FIT
   │
   ▼
Vocabulary + IDF
   │
   ▼
Save Vectorizer
```

Then production:

```text
New Incident
     │
     ▼
Saved Vectorizer
     │
     ▼
Transform
     │
     ▼
Numerical Vector
     │
     ▼
Saved ML Model
     │
     ▼
Prediction
```

This is why your `FeatureEngineer` class is not just a convenience class. It is becoming an important component of the **ML inference pipeline**.

And when you eventually move SIRA to **AWS/SageMaker**, this same principle remains: the preprocessing/vectorization artifacts used during training must be preserved and reused consistently during inference.